In [0]:
# ==================================================
# TSI - FASE 3: ANÁLISIS EXPLORATORIO
# PASO 44: PERFIL GENERAL DEL DATASET
# ==================================================

from pyspark.sql import functions as F

TABLA_TSI = "workspace.tsi.international_observations"

TOTAL_ESPERADO = 41_770_880

df_tsi = spark.table(TABLA_TSI)

print("=== INFORMACIÓN GENERAL ===")

print(f"Tabla: {TABLA_TSI}")

total = df_tsi.count()
columnas = len(df_tsi.columns)

print(f"Registros: {total:,}")
print(f"Columnas: {columnas}")

assert total == TOTAL_ESPERADO
assert columnas == 28

print("\n=== COBERTURA TEMPORAL POR FUENTE ===")

cobertura = (
    df_tsi
    .groupBy("ciudad", "fuente")
    .agg(
        F.count("*").alias("registros"),
        F.min("timestamp").alias("fecha_inicio"),
        F.max("timestamp").alias("fecha_fin"),
        F.countDistinct("segmento_id").alias("segmentos_distintos")
    )
    .orderBy("ciudad", "fuente")
)

cobertura.show(truncate=False)

print("\n=== DISTRIBUCIÓN DE CALIDAD ===")

calidad = (
    df_tsi
    .groupBy("fuente", "calidad_registro")
    .count()
    .orderBy("fuente", "calidad_registro")
)

calidad.show(100, truncate=False)

print("\n=== RESULTADO ===")
print("Perfil general del dataset generado correctamente.")

In [0]:
# ==================================================
# TSI - FASE 3: ANÁLISIS EXPLORATORIO
# PASO 45: DIAGNÓSTICO DE VALORES NULOS
# ==================================================

from pyspark.sql import functions as F

# Columnas numéricas disponibles
columnas_numericas = [
    campo.name
    for campo in df_tsi.schema.fields
    if campo.dataType.simpleString()
    in ("double", "int", "bigint", "float")
]

print("=== COLUMNAS NUMÉRICAS ===")
print(columnas_numericas)

# Contabilizar registros y nulos por fuente
expresiones = [
    F.count("*").alias("registros")
]

for columna in columnas_numericas:

    expresiones.append(
        F.sum(
            F.when(
                F.col(columna).isNull(),
                1
            ).otherwise(0)
        ).alias(f"nulos_{columna}")
    )

diagnostico = (
    df_tsi
    .groupBy("fuente")
    .agg(*expresiones)
    .orderBy("fuente")
)

# Convertir conteos a porcentajes para facilitar
# la comparación entre fuentes de distintos tamaños.
porcentajes = diagnostico.select(
    "fuente",
    "registros",
    *[
        F.round(
            F.col(f"nulos_{columna}")
            / F.col("registros") * 100,
            2
        ).alias(f"pct_nulos_{columna}")
        for columna in columnas_numericas
    ]
)

print("\n=== PORCENTAJE DE NULOS POR FUENTE ===")

porcentajes.show(
    truncate=False,
    vertical=True
)

print("\nDIAGNÓSTICO COMPLETADO")

In [0]:
# ==================================================
# TSI - FASE 3: ANÁLISIS EXPLORATORIO
# PASO 46: ESTADÍSTICAS DE MÉTRICAS PRINCIPALES
# ==================================================

from pyspark.sql import functions as F

METRICAS_PRINCIPALES = {
    "istanbul_traffic_index": "indice_congestion",
    "sao_paulo_cet": "congestion_longitud_m",
    "madrid_traffic_points": "flujo_veh_h",
    "metr_la": "valor_fuente",
    "pems_bay": "valor_fuente",
}

resultados = []

for fuente, metrica in METRICAS_PRINCIPALES.items():

    print(f"\n=== {fuente.upper()} ===")
    print(f"Métrica: {metrica}")

    df_fuente = (
        df_tsi
        .filter(F.col("fuente") == fuente)
        .select(F.col(metrica).alias("valor"))
    )

    estadisticas = df_fuente.agg(
        F.count("*").alias("registros"),
        F.count("valor").alias("valores_validos"),
        F.min("valor").alias("minimo"),
        F.max("valor").alias("maximo"),
        F.avg("valor").alias("promedio"),
        F.stddev("valor").alias("desviacion"),
        F.sum(
            F.when(F.col("valor") < 0, 1).otherwise(0)
        ).alias("valores_negativos"),
        F.sum(
            F.when(F.isnan("valor"), 1).otherwise(0)
        ).alias("valores_nan"),
    ).first()

    resultado = estadisticas.asDict()

    resultado["fuente"] = fuente
    resultado["metrica"] = metrica

    resultados.append(resultado)

    print(f"Registros: {resultado['registros']:,}")
    print(f"Valores válidos: {resultado['valores_validos']:,}")
    print(f"Mínimo: {resultado['minimo']}")
    print(f"Máximo: {resultado['maximo']}")
    print(f"Promedio: {resultado['promedio']}")
    print(f"Desviación estándar: {resultado['desviacion']}")
    print(f"Valores negativos: {resultado['valores_negativos']:,}")
    print(f"Valores NaN: {resultado['valores_nan']:,}")

print("\n=== RESULTADO ===")
print("Análisis estadístico completado.")

In [0]:
# ==================================================
# TSI - FASE 3: ANÁLISIS EXPLORATORIO
# PASO 47: PERCENTILES DE MÉTRICAS PRINCIPALES
# ==================================================

from pyspark.sql import functions as F

PERCENTILES = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]

for fuente, metrica in METRICAS_PRINCIPALES.items():

    print(f"\n{'=' * 55}")
    print(f"FUENTE: {fuente.upper()}")
    print(f"MÉTRICA: {metrica}")
    print(f"{'=' * 55}")

    df_fuente = (
        df_tsi
        .filter(F.col("fuente") == fuente)
        .select(F.col(metrica).alias("valor"))
        .where(F.col("valor").isNotNull())
    )

    percentiles = (
        df_fuente
        .agg(
            F.percentile_approx(
                "valor",
                PERCENTILES,
                10000
            ).alias("percentiles")
        )
        .first()["percentiles"]
    )

    for porcentaje, valor in zip(PERCENTILES, percentiles):

        print(
            f"P{int(porcentaje * 100):02d}: {valor:.4f}"
        )

print("\nANÁLISIS DE PERCENTILES COMPLETADO")

In [0]:
# ==================================================
# TSI - FASE 3: ANÁLISIS EXPLORATORIO
# PASO 48: DIAGNÓSTICO DE CEROS EN SENSORES
# ==================================================

from pyspark.sql import functions as F

FUENTES_VELOCIDAD = ["metr_la", "pems_bay"]

df_velocidades = (
    df_tsi
    .filter(F.col("fuente").isin(FUENTES_VELOCIDAD))
    .select(
        "fuente",
        "segmento_id",
        "timestamp",
        "valor_fuente"
    )
)

print("=== DISTRIBUCIÓN DE CEROS POR FUENTE ===")

resumen = (
    df_velocidades
    .groupBy("fuente")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("valor_fuente") == 0, 1).otherwise(0)
        ).alias("registros_cero"),
        F.countDistinct("segmento_id").alias("sensores")
    )
    .withColumn(
        "porcentaje_ceros",
        F.round(
            F.col("registros_cero") / F.col("registros") * 100,
            4
        )
    )
    .orderBy("fuente")
)

resumen.show(truncate=False)

print("\n=== SENSORES CON MÁS VALORES CERO ===")

ceros_por_sensor = (
    df_velocidades
    .groupBy("fuente", "segmento_id")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(F.col("valor_fuente") == 0, 1).otherwise(0)
        ).alias("ceros")
    )
    .withColumn(
        "porcentaje_ceros",
        F.round(
            F.col("ceros") / F.col("registros") * 100,
            2
        )
    )
)

(
    ceros_por_sensor
    .filter(F.col("ceros") > 0)
    .orderBy(
        F.desc("porcentaje_ceros"),
        F.desc("ceros")
    )
    .show(20, truncate=False)
)

print("\nDIAGNÓSTICO DE CEROS COMPLETADO")

In [0]:
# ==================================================
# TSI - FASE 3: ANÁLISIS EXPLORATORIO
# PASO 49: SECUENCIAS CONSECUTIVAS DE CEROS
# ==================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Analizamos exclusivamente METR-LA
df_metr = (
    df_tsi
    .filter(F.col("fuente") == "metr_la")
    .select(
        "segmento_id",
        "timestamp",
        "valor_fuente"
    )
)

# Orden temporal por sensor
ventana = Window.partitionBy("segmento_id").orderBy("timestamp")

# Identificar el valor y timestamp anteriores
df_secuencias = (
    df_metr
    .withColumn(
        "timestamp_anterior",
        F.lag("timestamp").over(ventana)
    )
    .withColumn(
        "valor_anterior",
        F.lag("valor_fuente").over(ventana)
    )
)

# Una nueva secuencia empieza cuando:
# - el valor actual es cero, y
# - el anterior no es cero o no está exactamente
#   a cinco minutos del registro actual.
df_secuencias = (
    df_secuencias
    .withColumn(
        "inicio_secuencia",
        F.when(
            (F.col("valor_fuente") == 0)
            & (
                F.col("valor_anterior").isNull()
                | (F.col("valor_anterior") != 0)
                | (
                    F.unix_timestamp("timestamp")
                    - F.unix_timestamp("timestamp_anterior")
                    != 300
                )
            ),
            1
        ).otherwise(0)
    )
)

# Asignar identificador a cada secuencia temporal
ventana_acumulada = (
    ventana.rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

df_secuencias = df_secuencias.withColumn(
    "id_secuencia",
    F.sum("inicio_secuencia").over(ventana_acumulada)
)

# Agrupar únicamente registros con valor cero
secuencias_cero = (
    df_secuencias
    .filter(F.col("valor_fuente") == 0)
    .groupBy("segmento_id", "id_secuencia")
    .agg(
        F.count("*").alias("registros_consecutivos"),
        F.min("timestamp").alias("inicio"),
        F.max("timestamp").alias("fin")
    )
    .withColumn(
        "duracion_minutos",
        F.col("registros_consecutivos") * 5
    )
)

print("=== SECUENCIAS MÁS LARGAS DE CEROS ===")

(
    secuencias_cero
    .orderBy(F.desc("registros_consecutivos"))
    .show(20, truncate=False)
)

print("\n=== RESUMEN GENERAL ===")

secuencias_cero.agg(
    F.count("*").alias("total_secuencias"),
    F.max("registros_consecutivos").alias("max_registros_consecutivos"),
    F.max("duracion_minutos").alias("max_duracion_minutos"),
    F.sum(
        F.when(
            F.col("registros_consecutivos") >= 12,
            1
        ).otherwise(0)
    ).alias("secuencias_60_min_o_mas")
).show(truncate=False)

print("\nANÁLISIS TEMPORAL COMPLETADO")

In [0]:
# ==================================================
# TSI - FASE 3: ANÁLISIS EXPLORATORIO
# PASO 50: SIMULTANEIDAD DE CEROS EN METR-LA
# ==================================================

from pyspark.sql import functions as F

df_metr = (
    df_tsi
    .filter(F.col("fuente") == "metr_la")
    .select("timestamp", "segmento_id", "valor_fuente")
)

# Número de sensores con cero en cada instante
simultaneidad = (
    df_metr
    .groupBy("timestamp")
    .agg(
        F.countDistinct("segmento_id").alias("sensores_presentes"),
        F.countDistinct(
            F.when(
                F.col("valor_fuente") == 0,
                F.col("segmento_id")
            )
        ).alias("sensores_cero")
    )
    .withColumn(
        "porcentaje_sensores_cero",
        F.round(
            F.col("sensores_cero")
            / F.col("sensores_presentes") * 100,
            2
        )
    )
)

print("=== INSTANTES CON MÁS SENSORES EN CERO ===")

(
    simultaneidad
    .orderBy(
        F.desc("sensores_cero"),
        F.asc("timestamp")
    )
    .show(20, truncate=False)
)

print("\n=== RESUMEN DE SIMULTANEIDAD ===")

simultaneidad.agg(
    F.count("*").alias("instantes_analizados"),
    F.max("sensores_cero").alias("max_sensores_cero"),
    F.max("porcentaje_sensores_cero").alias("max_porcentaje_cero"),
    F.sum(
        F.when(
            F.col("porcentaje_sensores_cero") >= 25,
            1
        ).otherwise(0)
    ).alias("instantes_25_pct_o_mas"),
    F.sum(
        F.when(
            F.col("porcentaje_sensores_cero") >= 50,
            1
        ).otherwise(0)
    ).alias("instantes_50_pct_o_mas")
).show(truncate=False)

print("\nANÁLISIS DE SIMULTANEIDAD COMPLETADO")

In [0]:
# ==================================================
# TSI - FASE 3: ANÁLISIS EXPLORATORIO
# PASO 51: CUANTIFICACIÓN DE CEROS SIMULTÁNEOS
# ==================================================

from pyspark.sql import functions as F

# Reutilizamos 'simultaneidad' del paso anterior.
# No modificamos ninguna tabla Delta.

print("=== DISTRIBUCIÓN DE SENSORES EN CERO ===")

distribucion_ceros = (
    simultaneidad
    .groupBy("sensores_cero")
    .agg(
        F.count("*").alias("instantes")
    )
    .orderBy(F.desc("sensores_cero"))
)

distribucion_ceros.show(30, truncate=False)

print("\n=== RESUMEN DE EVENTOS EXTREMOS ===")

resumen_extremos = simultaneidad.agg(
    F.count("*").alias("total_instantes"),

    F.sum(
        F.when(
            F.col("sensores_cero") == 207,
            1
        ).otherwise(0)
    ).alias("instantes_207_cero"),

    F.sum(
        F.when(
            F.col("sensores_cero") >= 104,
            1
        ).otherwise(0)
    ).alias("instantes_104_o_mas"),

    F.sum(
        F.when(
            F.col("sensores_cero") == 0,
            1
        ).otherwise(0)
    ).alias("instantes_sin_ceros")
)

resumen_extremos.show(truncate=False)

print("\n=== PORCENTAJE DE INSTANTES CON 207 CEROS ===")

resumen_extremos.select(
    F.round(
        F.col("instantes_207_cero")
        / F.col("total_instantes") * 100,
        4
    ).alias("porcentaje_instantes_207_cero")
).show(truncate=False)

print("\nDIAGNÓSTICO COMPLETADO")

In [0]:
# ==================================================
# TSI - FASE 3: ANÁLISIS EXPLORATORIO
# PASO 52: ETIQUETADO EXPERIMENTAL DE CALIDAD
# ==================================================

from pyspark.sql import functions as F

# Instantes donde los 207 sensores de METR-LA
# registraron exactamente cero.
instantes_cero_global = (
    simultaneidad
    .filter(
        (F.col("sensores_presentes") == 207)
        & (F.col("sensores_cero") == 207)
    )
    .select("timestamp")
    .distinct()
    .withColumn("cero_global", F.lit(True))
)

# Crear etiquetas derivadas sin alterar el dataset.
df_metr_calidad = (
    df_tsi
    .filter(F.col("fuente") == "metr_la")
    .join(
        instantes_cero_global,
        on="timestamp",
        how="left"
    )
    .withColumn(
        "calidad_derivada",
        F.when(
            F.col("cero_global") == True,
            F.lit("sospechoso_cero_global")
        )
        .when(
            F.col("valor_fuente") == 0,
            F.lit("cero_no_diagnosticado")
        )
        .otherwise(
            F.lit("sin_alerta_regla_qc_metr_01")
        )
    )
    .drop("cero_global")
)

print("=== DISTRIBUCIÓN DE ETIQUETAS ===")

resumen_calidad = (
    df_metr_calidad
    .groupBy("calidad_derivada")
    .count()
    .orderBy("calidad_derivada")
)

resumen_calidad.show(truncate=False)

print("\n=== VALIDACIÓN DE INTEGRIDAD ===")

total_original = (
    df_tsi
    .filter(F.col("fuente") == "metr_la")
    .count()
)

total_etiquetado = df_metr_calidad.count()

print(f"Registros originales: {total_original:,}")
print(f"Registros etiquetados: {total_etiquetado:,}")

assert total_original == total_etiquetado
assert total_original == 7_094_304

print("\n=== VALIDACIÓN DE REGLA QC-METR-01 ===")

conteos = {
    fila["calidad_derivada"]: fila["count"]
    for fila in resumen_calidad.collect()
}

assert conteos.get("sospechoso_cero_global", 0) == 444_636
assert conteos.get("cero_no_diagnosticado", 0) == 130_666
assert sum(conteos.values()) == total_original

print("Regla aplicada correctamente.")
print("Los valores originales permanecen intactos.")
print("No se ha escrito ninguna tabla Delta.")

In [0]:
# ==================================================
# TSI - FASE 3: CONTROL DE CALIDAD
# PASO 53: PERSISTENCIA CORREGIDA EN DELTA LAKE
# ==================================================

TABLA_CALIDAD = "workspace.tsi.metr_la_calidad"

TOTAL_ESPERADO = 7_094_304

CONTEOS_ESPERADOS = {
    "sospechoso_cero_global": 444_636,
    "cero_no_diagnosticado": 130_666,
    "sin_alerta_regla_qc_metr_01": 6_519_002,
}

print("=== VALIDACIÓN PREVIA ===")

total = df_metr_calidad.count()

assert total == TOTAL_ESPERADO

conteos = {
    fila["calidad_derivada"]: fila["count"]
    for fila in (
        df_metr_calidad
        .groupBy("calidad_derivada")
        .count()
        .collect()
    )
}

assert conteos == CONTEOS_ESPERADOS

print("Integridad de las etiquetas confirmada.")

print("\n=== PERSISTENCIA EN DELTA LAKE ===")

if spark.catalog.tableExists(TABLA_CALIDAD):

    print("La tabla ya existe.")
    print("No se realizará ninguna escritura.")

else:

    print("Creando tabla Delta de calidad...")

    (
        df_metr_calidad
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_CALIDAD)
    )

    print("Tabla Delta creada correctamente.")

print(f"Tabla: {TABLA_CALIDAD}")

print("\n=== VERIFICACIÓN DE TABLA PERSISTENTE ===")

df_guardado = spark.table(TABLA_CALIDAD)

total_guardado = df_guardado.count()

print(f"Registros guardados: {total_guardado:,}")

assert total_guardado == TOTAL_ESPERADO

distribucion_guardada = {
    fila["calidad_derivada"]: fila["count"]
    for fila in (
        df_guardado
        .groupBy("calidad_derivada")
        .count()
        .collect()
    )
}

assert distribucion_guardada == CONTEOS_ESPERADOS

detalle = spark.sql(
    f"DESCRIBE DETAIL {TABLA_CALIDAD}"
)

detalle.select(
    "format",
    "numFiles"
).show(truncate=False)

assert detalle.select("format").first()[0] == "delta"

print("\n=== RESULTADO FINAL ===")
print("Persistencia validada correctamente.")
print("Las tablas originales permanecen intactas.")

# Control de calidad experimental — METR-LA

## Regla QC-METR-01: Ceros simultáneos

### Objetivo
Identificar observaciones potencialmente afectadas por problemas de adquisición o representación de datos en METR-LA.

### Evidencia observada
- Registros analizados: 7,094,304.
- Sensores: 207.
- Registros con valor cero: 575,302.
- Instantes con los 207 sensores en cero: 2,148.
- Observaciones asociadas a esos instantes: 444,636.
- Porcentaje de observaciones etiquetadas: aproximadamente 6.27 %.

### Criterio experimental
Una observación se etiqueta como `sospechoso_cero_global` cuando los 207 sensores registran simultáneamente un valor igual a cero.

Los demás ceros se clasifican como `cero_no_diagnosticado`.

Las observaciones que no activan esta regla reciben la etiqueta `sin_alerta_regla_qc_metr_01`.

### Implementación
- Tabla original: `workspace.tsi.metr_la_clean`.
- Tabla derivada: `workspace.tsi.metr_la_calidad`.
- Tecnología: Apache Spark y Delta Lake.
- No se eliminaron registros.
- No se modificaron los valores originales.

### Limitaciones
La simultaneidad de ceros constituye una señal de calidad sospechosa, no una prueba definitiva de falla de sensores.

Los ceros restantes requieren análisis adicional.

La regla no determina si ocurrió congestión real y no debe utilizarse como etiqueta de congestión.

### Aplicación futura
Comparar experimentos de modelado con todos los registros frente a experimentos que excluyan las observaciones sospechosas.

Registrar en MLflow las reglas de filtrado, los parámetros y las métricas de evaluación.